# VPN-Sentinel: Professional End-to-End Notebook

This notebook captures the full VPN-Sentinel workflow: synthetic data generation, exploratory analysis, modeling, robustness testing, explainability, and deployment readiness.

## 1. Project Scope & Setup

The project covers:
- Flow-level VPN vs non-VPN detection
- VPN protocol identification (OpenVPN, WireGuard, IKEv2)
- Browser-level multi-signal classification
- Adversarial traffic shaping evaluation
- SHAP explanation and deployment integration

First, let's import the required libraries and set up our working environment.

In [ ]:
import os
import warnings
from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import shap
warnings.filterwarnings('ignore')
np.random.seed(42)
plt.style.use('seaborn-v0_8')
ROOT = Path.cwd()
DATA_DIR = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)
print(f'Working directory: {ROOT}')


## 2. Data Generation & Loading

We generate synthetic network flows and browser flows if they do not already exist.
We also apply adversarial traffic shaping to test our model's robustness later.

In [ ]:
from data_generator import generate_flows, generate_browser_flows
from adversarial_shaper import apply_evasion
train_flow_path = DATA_DIR / 'train_flows.csv'
test_flow_path = DATA_DIR / 'test_flows.csv'
train_browser_path = DATA_DIR / 'train_browser_flows.csv'
test_browser_path = DATA_DIR / 'test_browser_flows.csv'
adv_test_path = DATA_DIR / 'test_flows_adversarial.csv'
if not all([train_flow_path.exists(), test_flow_path.exists(), train_browser_path.exists(), test_browser_path.exists(), adv_test_path.exists()]):
    print('Generating fresh datasets...')
    flow_df = generate_flows(12000)
    flow_train, flow_test = train_test_split(flow_df, test_size=0.2, random_state=42, stratify=flow_df['is_vpn'])
    flow_train.to_csv(train_flow_path, index=False)
    flow_test.to_csv(test_flow_path, index=False)
    browser_df = generate_browser_flows(8000)
    br_train, br_test = train_test_split(browser_df, test_size=0.2, random_state=42, stratify=browser_df['is_vpn'])
    br_train.to_csv(train_browser_path, index=False)
    br_test.to_csv(test_browser_path, index=False)
    adv_test = apply_evasion(flow_test)
    adv_test.to_csv(adv_test_path, index=False)
train_df = pd.read_csv(train_flow_path)
test_df = pd.read_csv(test_flow_path)
train_br_df = pd.read_csv(train_browser_path)
test_br_df = pd.read_csv(test_browser_path)
test_adv_df = pd.read_csv(adv_test_path)
print(f'Flow train shape: {train_df.shape}')
print(f'Flow test shape: {test_df.shape}')
print(f'Browser train shape: {train_br_df.shape}')


## 3. Exploratory Data Analysis (EDA)

Let's visualize the distribution of labels in our datasets to understand the class balance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
train_df['is_vpn'].value_counts().rename({0: 'Non-VPN', 1: 'VPN'}).plot(kind='bar', ax=axes[0], color=['#4C78A8', '#F58518'])
axes[0].set_title('Flow Label Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
train_br_df['is_vpn'].value_counts().rename({0: 'Non-VPN', 1: 'VPN'}).plot(kind='bar', ax=axes[1], color=['#54A24B', '#EECA3B'])
axes[1].set_title('Browser Label Distribution')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()


## 4. Feature Engineering

We select the relevant features for our models and handle any missing or non-numeric values.
We will prepare data for two stages:
- **Stage 1**: VPN vs Non-VPN detection
- **Stage 2**: VPN Protocol classification (OpenVPN, WireGuard, IKEv2)

In [ ]:
flow_features = [
    'duration', 'fwd_pkt_len_mean', 'bwd_pkt_len_mean',
    'flow_iat_mean', 'flow_iat_std', 'jitter_ratio', 'packets_per_sec', 'bytes_per_sec',
    'fwd_pkt_len_max', 'fwd_pkt_len_min', 'bwd_pkt_len_max', 'bwd_pkt_len_min',
    'fwd_pkt_len_std', 'bwd_pkt_len_std', 'flow_iat_max', 'flow_iat_min'
]
browser_features = [
    'duration', 'flow_iat_mean', 'flow_iat_std', 'jitter_ratio', 'packets_per_sec',
    'webrtc_ip_mismatch', 'webrtc_blocked', 'timezone_mismatch_score', 'language_mismatch_score',
    'geo_ip_distance_km', 'has_geo_permission', 'is_datacenter_ip', 'is_known_vpn_ip', 'proxy_header_detected'
]
for df in [train_df, test_df, test_adv_df, train_br_df, test_br_df]:
    for col in flow_features + browser_features:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(-1.0)
X_train_s1 = train_df[flow_features]
y_train_s1 = train_df['is_vpn']
X_test_s1 = test_df[flow_features]
y_test_s1 = test_df['is_vpn']
vpn_train = train_df[train_df['is_vpn'] == 1].copy()
vpn_test = test_df[test_df['is_vpn'] == 1].copy()
X_train_s2 = vpn_train[flow_features]
y_train_s2 = vpn_train['vpn_protocol']
X_test_s2 = vpn_test[flow_features]
y_test_s2 = vpn_test['vpn_protocol']
X_train_br = train_br_df[browser_features]
y_train_br = train_br_df['is_vpn']
X_test_br = test_br_df[browser_features]
y_test_br = test_br_df['is_vpn']
print('Feature preparation complete.')


## 5. Model Training & Evaluation

First, we analyze the learning curves to see how the model performance grows with the amount of training data.

In [ ]:
def evaluate_growth(X_train, y_train, X_test, y_test, train_sizes):
    """Evaluates model performance at different training set sizes."""
    rows = []
    for size in train_sizes:
        subset_idx = np.random.choice(len(X_train), size=size, replace=False)
        X_sub = X_train.iloc[subset_idx]
        y_sub = y_train.iloc[subset_idx]
        model = RandomForestClassifier(n_estimators=180, max_depth=12, random_state=42, n_jobs=-1)
        model.fit(X_sub, y_sub)
        pred = model.predict(X_test)
        rows.append({
            'train_size': size, 
            'accuracy': accuracy_score(y_test, pred), 
            'f1': f1_score(y_test, pred, average='weighted')
        })
    return pd.DataFrame(rows)
train_sizes = [250, 500, 1000, 2500, 5000]
growth_s1 = evaluate_growth(X_train_s1, y_train_s1, X_test_s1, y_test_s1, train_sizes)
growth_s2 = evaluate_growth(X_train_s2, y_train_s2, X_test_s2, y_test_s2, train_sizes)
print("Stage 1 Growth:")
print(growth_s1.to_string(index=False))
print('\nStage 2 Growth:')
print(growth_s2.to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(growth_s1['train_size'], growth_s1['accuracy'], marker='o', label='Accuracy')
axes[0].plot(growth_s1['train_size'], growth_s1['f1'], marker='s', label='F1')
axes[0].set_title('Stage 1 Learning Curve (VPN vs Non-VPN)')
axes[0].set_xlabel('Training Set Size')
axes[0].set_ylabel('Score')
axes[0].legend()
axes[1].plot(growth_s2['train_size'], growth_s2['accuracy'], marker='o', label='Accuracy')
axes[1].plot(growth_s2['train_size'], growth_s2['f1'], marker='s', label='F1')
axes[1].set_title('Stage 2 Learning Curve (Protocols)')
axes[1].set_xlabel('Training Set Size')
axes[1].set_ylabel('Score')
axes[1].legend()
plt.tight_layout()
plt.show()


Next, we train the final models on the full training sets and evaluate their performance. We also export the trained models and feature lists for deployment.

In [ ]:
model_s1 = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
model_s1.fit(X_train_s1, y_train_s1)
pred_s1 = model_s1.predict(X_test_s1)
print('=== Stage 1 Report (VPN vs Non-VPN) ===')
print(classification_report(y_test_s1, pred_s1, target_names=['Non-VPN', 'VPN']))
model_s2 = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
model_s2.fit(X_train_s2, y_train_s2)
pred_s2 = model_s2.predict(X_test_s2)
print('\n=== Stage 2 Report (VPN Protocol) ===')
print(classification_report(y_test_s2, pred_s2, target_names=['OpenVPN', 'WireGuard', 'IKEv2']))
model_br_s1 = RandomForestClassifier(n_estimators=250, random_state=42, n_jobs=-1)
model_br_s1.fit(X_train_br, y_train_br)
pred_br_s1 = model_br_s1.predict(X_test_br)
print('\n=== Browser Model Report ===')
print(classification_report(y_test_br, pred_br_s1, target_names=['Non-VPN', 'VPN']))
joblib.dump(model_s1, MODELS_DIR / 'stage1_model.pkl')
joblib.dump(model_s2, MODELS_DIR / 'stage2_model.pkl')
joblib.dump(model_br_s1, MODELS_DIR / 'browser_stage1_model.pkl')
joblib.dump(flow_features, MODELS_DIR / 'features.pkl')
joblib.dump(browser_features, MODELS_DIR / 'timing_features.pkl')
print('Models and features saved successfully.')


## 6. Adversarial Robustness

We test our trained Stage 1 model against adversarial traffic where malicious actors try to hide VPN signatures by shaping traffic.

In [ ]:
X_test_adv_s1 = test_adv_df[flow_features]
y_test_adv_s1 = test_adv_df['is_vpn']
pred_adv_s1 = model_s1.predict(X_test_adv_s1)
clean_acc = accuracy_score(y_test_s1, pred_s1)
adv_acc = accuracy_score(y_test_adv_s1, pred_adv_s1)
print(f'Clean accuracy: {clean_acc:.4f}')
print(f'Adversarial accuracy: {adv_acc:.4f}')
print(f'Performance drop: {clean_acc - adv_acc:.4f}')


## 7. Model Explainability

Finally, we use SHAP (SHapley Additive exPlanations) to understand which features are most important for our model's predictions.

In [ ]:
explainer_s1 = shap.TreeExplainer(model_s1)
sample_df = X_test_s1.iloc[:80]
shap_values_s1 = explainer_s1.shap_values(sample_df)
if isinstance(shap_values_s1, list):
    shap_values_s1 = shap_values_s1[1]
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_s1, sample_df, plot_type='bar', show=False)
plt.title('SHAP Feature Importance for VPN Detection (Stage 1)')
plt.tight_layout()
plt.show()


## Conclusion

This notebook presents a professional, end-to-end view of the VPN-Sentinel project. It combines data generation, learning-curve analysis, robust evaluation, SHAP explainability, and deployment-ready artifact export.